In [1]:
# Relative paths - portable across machines after cloning the repository
from pathlib import Path

DATA_RAW       = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")


---

## Airline Reviews — Full Cleaning Pipeline

*Source notebook: `notebook_airline review.ipynb`*


In [3]:
import re
import pandas as pd
import pycountry

# ==============================================================================
# BƯỚC 0: TẢI DỮ LIỆU & CHUẨN BỊ TỪ ĐIỂN
# ==============================================================================
df = pd.read_csv(DATA_RAW / "airline_reviews.csv")
print(f"Kích thước file gốc: {df.shape}")

# 1. Tự tạo danh sách Stopwords mở rộng (Bộ lọc từ khóa siêu sạch cho Hàng không)
base_stopwords = {
    "i",
    "me",
    "my",
    "myself",
    "we",
    "our",
    "ours",
    "ourselves",
    "you",
    "your",
    "yours",
    "he",
    "him",
    "his",
    "she",
    "her",
    "it",
    "its",
    "they",
    "them",
    "their",
    "what",
    "which",
    "who",
    "whom",
    "this",
    "that",
    "these",
    "those",
    "am",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "have",
    "has",
    "had",
    "having",
    "do",
    "does",
    "did",
    "doing",
    "a",
    "an",
    "the",
    "and",
    "but",
    "if",
    "or",
    "because",
    "as",
    "until",
    "while",
    "of",
    "at",
    "by",
    "for",
    "with",
    "about",
    "against",
    "between",
    "into",
    "through",
    "during",
    "before",
    "after",
    "above",
    "below",
    "to",
    "from",
    "up",
    "down",
    "in",
    "out",
    "on",
    "off",
    "over",
    "under",
    "again",
    "further",
    "then",
    "once",
    "here",
    "there",
    "when",
    "where",
    "why",
    "how",
    "all",
    "any",
    "both",
    "each",
    "few",
    "more",
    "most",
    "other",
    "some",
    "such",
    "no",
    "nor",
    "not",
    "only",
    "own",
    "same",
    "so",
    "than",
    "too",
    "very",
    "can",
    "will",
    "just",
    "don",
    "should",
    "now",
}

# Các từ đặc thù hàng không & từ trung tính gây nhiễu Word Cloud (đã bổ sung từ bạn nhận xét)
neutral_noise_words = {
    "flight",
    "flights",
    "airline",
    "airlines",
    "flew",
    "plane",
    "planes",
    "aircraft",
    "would",
    "could",
    "get",
    "got",
    "getting",
    "one",
    "two",
    "three",
    "time",
    "times",
    "passenger",
    "passengers",
    "also",
    "even",
    "really",
    "first",
    "second",
    "day",
    "days",
    "bit",
    "much",
    "give",
    "gave",
    "see",
    "saw",
    "seen",
    "us",
    "take",
    "took",
    "taking",
    "hour",
    "hours",
    "minute",
    "minutes",
    "around",
    "since",
    "another",
    "someone",
    "anyone",
    "something",
    "anything",
    "nothing",
    "everything",
    "say",
    "said",
    "told",
    "ask",
    "asked",
    "went",
    "go",
    "going",
    "come",
    "came",
    "back",
    "next",
    "last",
    "many",
    "lot",
    "lots",
    "little",
    "small",
    "big",
    "know",
    "think",
    "thought",
    "want",
    "wanted",
    "need",
    "needed",
    "use",
    "used",
    "using",
    "try",
    "tried",
    "still",
    "like",
    "quite",
    "well",
    "way",
    "make",
    "made",
    "good",
    "bad",  # Bỏ good/bad chung chung để lòi ra các từ cụ thể như: rude, delicious, delay, comfortable
}
stop_words = base_stopwords.union(neutral_noise_words)

# 2. Chuẩn bị tập từ điển quốc gia chuẩn ISO
valid_countries = {c.name.lower() for c in pycountry.countries}
for c in pycountry.countries:
    if hasattr(c, "official_name"):
        valid_countries.add(c.official_name.lower())
    if hasattr(c, "common_name"):
        valid_countries.add(c.common_name.lower())
custom_countries = {
    "uk",
    "usa",
    "united states",
    "united kingdom",
    "uae",
    "russia",
    "south korea",
    "north korea",
    "vietnam",
    "laos",
    "taiwan",
    "hong kong",
    "macau",
}
valid_countries = valid_countries.union(custom_countries)

Kích thước file gốc: (156323, 26)


C:\Users\luong\AppData\Local\Temp\ipykernel_17216\483355370.py:8: DtypeWarning: Columns (14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"../data/raw/airline_reviews.csv")


In [4]:
# ==============================================================================
# BƯỚC 1: XÓA CỘT THỪA & CHUYỂN DATA TYPE CHUẨN
# ==============================================================================
print("1. Đang xử lý cột và Data Type...")
cols_to_drop = ["customer_name", "updated_at", "date_flown"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Chuyển ngày tháng
df["date_submitted"] = pd.to_datetime(df["date_submitted"], errors="coerce")

# Chuyển điểm số về dạng Numeric
rating_cols = [
    "seat_comfort",
    "cabin_staff_service",
    "food_and_beverages",
    "inflight_entertainment",
    "ground_service",
    "wifi_and_connectivity",
    "value_for_money",
    "recommended",
]
for col in rating_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

1. Đang xử lý cột và Data Type...


In [5]:
# ==============================================================================
# BƯỚC 2: GIẢI QUYẾT VẤN ĐỀ LỊCH SỬ DỮ LIỆU (HISTORICAL GAP)
# ==============================================================================
# Tạo cột 'data_era' để Power BI dễ làm bộ lọc Slicer
df["data_era"] = df["date_submitted"].apply(
    lambda x: "Modern (2015-Present)"
    if pd.notnull(x) and x.year >= 2015
    else "Historical (Pre-2015)"
)

In [6]:
# ==============================================================================
# BƯỚC 3: CHUẨN HÓA VÀ GOM NHÓM MÁY BAY (AIRCRAFT GROUPING)
# ==============================================================================
def clean_aircraft(val):
    if pd.isna(val):
        return "Unknown"
    val_str = str(val).upper()

    # Nếu có dấu / hoặc AND -> Khách bay nối chuyến nhiều loại máy bay
    if "/" in val_str or " AND " in val_str or "," in val_str:
        return "Mixed Fleet / Multiple"

    # Nhóm Boeing
    if "BOEING" in val_str or re.search(r"\bB?7[0-9]{2}\b", val_str):
        match = re.search(r"7[0-9]{2}", val_str)
        return f"Boeing {match.group(0)}" if match else "Boeing (Other)"

    # Nhóm Airbus
    elif "AIRBUS" in val_str or re.search(r"\bA3[0-9]{2}\b", val_str):
        match = re.search(r"3[0-9]{2}", val_str)
        return f"Airbus A{match.group(0)}" if match else "Airbus (Other)"

    # Nhóm khác
    elif any(x in val_str for x in ["EMBRAER", "ERJ", "E1", "E2"]):
        return "Embraer"
    elif "ATR" in val_str:
        return "ATR"
    else:
        return "Other Aircraft"


print("2. Đang gom nhóm loại máy bay...")
if "aircraft" in df.columns:
    df["aircraft"] = df["aircraft"].apply(clean_aircraft)

2. Đang gom nhóm loại máy bay...


In [7]:
# ==============================================================================
# BƯỚC 4: CHUẨN HÓA QUỐC TỊCH (NATIONALITY CLEANING)
# ==============================================================================
def clean_nationality(val):
    if pd.isna(val):
        return "Unknown"
    clean_val = str(val).strip().lower()
    return str(val).strip().title() if clean_val in valid_countries else "Unknown"


print("3. Đang làm sạch cột quốc tịch...")
if "nationality" in df.columns:
    df["nationality"] = df["nationality"].apply(clean_nationality)

3. Đang làm sạch cột quốc tịch...


In [8]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

# Tải bộ từ điển cảm xúc của VADER (Chỉ cần tải 1 lần)
nltk.download("vader_lexicon", quiet=True)

# Khởi tạo công cụ phân tích
sia = SentimentIntensityAnalyzer()

# ==============================================================================
# BƯỚC 4.5: PHÂN TÍCH TRẠNG THÁI CẢM XÚC (SENTIMENT ANALYSIS)
# ==============================================================================
print("Đang chấm điểm cảm xúc (Sentiment) cho các bài review...")


def get_sentiment_label(text):
    if pd.isna(text) or str(text).strip() == "":
        return "Neutral", 0.0

    # Tính điểm compound (từ -1.0 đến 1.0)
    score = sia.polarity_scores(str(text))["compound"]

    # Phân loại theo ngưỡng chuẩn quốc tế của VADER
    if score >= 0.05:
        return "Positive", score
    elif score <= -0.05:
        return "Negative", score
    else:
        return "Neutral", score


# Áp dụng hàm để tạo 2 cột mới:
# 1. 'sentiment_label': Nhãn (Positive / Negative / Neutral) để vẽ biểu đồ tròn/cột
# 2. 'sentiment_score': Điểm số cụ thể (-1 đến 1) để tính trung bình KPI
sentiment_results = df["review"].apply(get_sentiment_label)
df["sentiment_label"] = [res[0] for res in sentiment_results]
df["sentiment_score"] = [res[1] for res in sentiment_results]

# Check thử kết quả
print(df[["review", "sentiment_label", "sentiment_score"]].head(5))

# ---> SAU BƯỚC NÀY BẠN MỚI TIẾP TỤC BƯỚC 5: TẠO CLEAN_KEYWORDS VÀ XÓA CỘT REVIEW GỐC <---

Đang chấm điểm cảm xúc (Sentiment) cho các bài review...
                                              review sentiment_label  \
0  I flew Lan Peru on a domestic flight from Lima...        Positive   
1  I flew Lan Peru on a domestic flight from Lima...        Positive   
2  Skymark is a low-fare airline in Japan. I flow...        Positive   
3  I have flown LanPeru into and out of Cuzco man...        Positive   
4  I have flown LanPeru into and out of Cuzco man...        Positive   

   sentiment_score  
0           0.8993  
1           0.8993  
2           0.4012  
3           0.5296  
4           0.5296  


In [9]:
# ==============================================================================
# BƯỚC 5: TÁCH TỪ KHÓA LÀM SẠCH TRIỆT ĐỂ CHO WORD CLOUD
# ==============================================================================
def extract_super_clean_keywords(text):
    if pd.isna(text):
        return ""
    # Chỉ lấy từ có từ 3 chữ cái trở lên
    words = re.findall(r"\b[a-z]{3,}\b", str(text).lower())
    # Lọc bỏ từ dừng mở rộng
    keywords = [w for w in words if w not in stop_words]
    return " ".join(keywords)


print("4. Đang tách từ khóa (siêu sạch) từ cột review...")
df["clean_keywords"] = df["review"].apply(extract_super_clean_keywords)

# Xóa cột review gốc bị nặng file
df = df.drop(columns=["review"])

4. Đang tách từ khóa (siêu sạch) từ cột review...


In [10]:
# Chuẩn hóa riêng cho nối chuyến trong file airline_reviews
for col in ["transit_city", "transit_airport"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace(["nan", "None", "NULL", "", "N/A"], "Direct Flight")
            .fillna("Direct Flight")
        )

In [11]:
city_cols = [
      "origin_city",
      "destination_city",
      "origin_airport",
      "destination_airport",
  ]
for col in city_cols:
      if col in df.columns:
          df[col] = (
              df[col]
              .astype(str)
              .str.strip()
              .str.title()
              .replace(["Nan", "None", "Null", ""], "Unknown")
          )

In [12]:
# ==============================================================================
# BƯỚC 6: XUẤT FILE HOÀN THIỆN
# ==============================================================================
output_filename = "airline_reviews_PBI_Master_Optimized_v2.csv"
df.to_csv(output_filename, index=False)
print("-" * 50)
print(f"XỬ LÝ HOÀN TẤT! File: '{output_filename}'")
print(f"Kích thước cuối cùng: {df.shape}")
print("-" * 50)
print("Thống kê nhanh cột Máy bay sau khi gom nhóm:")
if "aircraft" in df.columns:
    print(df["aircraft"].value_counts().head(10))

--------------------------------------------------
XỬ LÝ HOÀN TẤT! File: 'airline_reviews_PBI_Master_Optimized_v2.csv'
Kích thước cuối cùng: (156323, 26)
--------------------------------------------------
Thống kê nhanh cột Máy bay sau khi gom nhóm:
aircraft
Unknown        115094
Boeing 737       7178
Airbus A320      6889
Boeing 777       6035
Airbus A330      4528
Boeing 787       4192
Airbus A321      2399
Airbus A380      2269
Airbus A350      1513
Airbus A319      1481
Name: count, dtype: int64


In [13]:
def final_quality_check(df, file_name):
    print(f"=== KIỂM TRA CHẤT LƯỢNG CUỐI CÙNG: {file_name} ===")

    # 1. Kiểm tra xem còn cột nào bị NULL/NaN không
    null_counts = df.isnull().sum()
    null_cols = null_counts[null_counts > 0]
    if not null_cols.empty:
        print("⚠️ Cảnh báo: Vẫn còn cột chứa giá trị NULL:")
        for col, count in null_cols.items():
            print(f"   - {col}: trống {count} dòng ({count/len(df)*100:.1f}%)")
    else:
        print("✔️ Tuyệt vời! Không còn ô NULL nào trong dữ liệu.")

    # 2. Kiểm tra kiểu dữ liệu của date_submitted và recommended
    if "date_submitted" in df.columns:
        print(f"✔️ Kiểu dữ liệu ngày tháng: {df['date_submitted'].dtype}")
    if "recommended" in df.columns:
        print(
            f"✔️ Kiểu dữ liệu NPS (recommended): {df['recommended'].dtype} (Giá trị mẫu: {df['recommended'].unique()})"
        )
    print("-" * 50)


# Ví dụ cách dùng:
final_quality_check(df, "Airline Reviews")
# final_quality_check(df_lounge, "Lounge Reviews")

=== KIỂM TRA CHẤT LƯỢNG CUỐI CÙNG: Airline Reviews ===
⚠️ Cảnh báo: Vẫn còn cột chứa giá trị NULL:
   - type_of_traveller: trống 39446 dòng (25.2%)
   - seat_type: trống 3024 dòng (1.9%)
   - seat_comfort: trống 16225 dòng (10.4%)
   - cabin_staff_service: trống 16593 dòng (10.6%)
   - food_and_beverages: trống 44381 dòng (28.4%)
   - inflight_entertainment: trống 65404 dòng (41.8%)
   - ground_service: trống 43556 dòng (27.9%)
   - wifi_and_connectivity: trống 115123 dòng (73.6%)
   - value_for_money: trống 2417 dòng (1.5%)
✔️ Kiểu dữ liệu ngày tháng: datetime64[ns]
✔️ Kiểu dữ liệu NPS (recommended): int64 (Giá trị mẫu: [0 1])
--------------------------------------------------


In [14]:
import pandas as pd

# =====================================================================
# BƯỚC 0: CẤU HÌNH TÊN FILE (Bạn có thể sửa tên file tại đây nếu cần)
# =====================================================================
file_input = DATA_PROCESSED / "airline_reviews_PBI_Master_Optimized_v2.csv"
file_output = "airline_reviews_PBI_Master_Optimized_V3.csv"

print(f"✈️ === BẮT ĐẦU VÁ LỖI CHO AIRLINE REVIEWS: {file_input} ===")

try:
    df = pd.read_csv(file_input)
    print(f"✔️ Đã đọc file thành công! Kích thước ban đầu: {df.shape}")
except FileNotFoundError:
    print(
        f"❌ Lỗi: Không tìm thấy file '{file_input}'. Vui lòng kiểm tra lại đường dẫn!"
    )
    exit()

# =====================================================================
# BƯỚC 1: VÁ LỖI CÁC CỘT PHÂN LOẠI (TEXT) -> KHÔNG ĐỂ TRỐNG (NULL)
# =====================================================================
print("1. Đang chuẩn hóa các cột phân loại và văn bản...")

# 1.1. Các cột phân loại chung -> Điền "Unknown"
general_text_cols = [
    "type_of_traveller",
    "seat_type",
    "aircraft",
    "nationality",
]
for col in general_text_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace(["nan", "None", "NULL", "", "N/A", "n/a"], "Unknown")
            .fillna("Unknown")
        )

# 1.2. Các cột địa điểm (Thành phố / Sân bay đi và đến) -> Điền "Unknown" & Viết hoa
city_cols = [
    "origin_city",
    "destination_city",
    "origin_airport",
    "destination_airport",
]
for col in city_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.title()
            .replace(["Nan", "None", "Null", "", "N/A", "N/a"], "Unknown")
            .fillna("Unknown")
        )

# 1.3. CỘT ĐẶC THÙ AIRLINE: Nối chuyến (Transit) -> Điền "Direct Flight" (Bay thẳng)
transit_cols = ["transit_city", "transit_airport"]
for col in transit_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace(["nan", "None", "NULL", "", "N/A", "n/a"], "Direct Flight")
            .fillna("Direct Flight")
        )

# =====================================================================
# BƯỚC 2: XÓA DÒNG RÁC (KHÁCH KHÔNG CHẤM BẤT KỲ ĐIỂM SỐ NÀO)
# =====================================================================
print("2. Đang kiểm tra và lọc bỏ các bài review rác...")

# 8 cột điểm số dịch vụ đặc thù của Hãng hàng không + cột value_for_money
airline_rating_cols = [
    "seat_comfort",
    "cabin_staff_service",
    "food_and_beverages",
    "inflight_entertainment",
    "ground_service",
    "wifi_and_connectivity",
    "value_for_money",
]

# Chỉ lấy các cột thực sự tồn tại trong file
existing_ratings = [col for col in airline_rating_cols if col in df.columns]

if existing_ratings:
    initial_rows = len(df)
    # Xóa dòng nếu TẤT CẢ các cột điểm số trên đều bị trống (NaN)
    df = df.dropna(subset=existing_ratings, how="all")
    deleted_rows = initial_rows - len(df)
    if deleted_rows > 0:
        print(
            f"✔️ Đã loại bỏ {deleted_rows} dòng (do khách không chấm bất kỳ điểm dịch vụ nào)."
        )
    else:
        print("✔️ Không có bài review rác nào cần loại bỏ.")

# =====================================================================
# BƯỚC 3: BÁO CÁO KIỂM TRA CHẤT LƯỢNG CUỐI CÙNG (FINAL AUDIT)
# =====================================================================
print("\n" + "=" * 55)
print("📊 BÁO CÁO CHẤT LƯỢNG DỮ LIỆU AIRLINE REVIEWS")
print("=" * 55)

# Kiểm tra NULL
null_counts = df.isnull().sum()
null_cols = null_counts[null_counts > 0]

if not null_cols.empty:
    print(
        "⚠️ Cảnh báo: Các cột sau vẫn còn ô NULL (CHỈ NÊN LÀ CỘT ĐIỂM SỐ RATING):"
    )
    for col, count in null_cols.items():
        ratio = (count / len(df)) * 100
        print(f"   - {col}: trống {count} dòng ({ratio:.1f}%)")
else:
    print("🎉 Tuyệt vời! 100% các ô trong bảng dữ liệu đều đã đầy đủ.")

# Kiểm tra kiểu dữ liệu quan trọng cho PBI
if "date_submitted" in df.columns:
    # Đảm bảo đúng chuẩn Datetime
    df["date_submitted"] = pd.to_datetime(
        df["date_submitted"], errors="coerce"
    )
    print(f"✔️ Kiểu dữ liệu ngày tháng: {df['date_submitted'].dtype}")

if "recommended" in df.columns:
    print(
        f"✔️ Kiểu dữ liệu NPS (recommended): {df['recommended'].dtype} (Các giá trị hiện có: {df['recommended'].unique()})"
    )

# =====================================================================
# BƯỚC 4: XUẤT FILE HOÀN THIỆN CHO POWER BI
# =====================================================================
df.to_csv(file_output, index=False)
print("-" * 55)
print(f"🚀 XỬ LÝ HOÀN TẤT! File chuẩn 100% đã lưu tại: '{file_output}'")
print(f"📐 Kích thước cuối cùng sẵn sàng lên Power BI: {df.shape}")
print("-" * 55)

✈️ === BẮT ĐẦU VÁ LỖI CHO AIRLINE REVIEWS: ../data/processed/airline_reviews_PBI_Master_Optimized_v2.csv ===
✔️ Đã đọc file thành công! Kích thước ban đầu: (156323, 26)
1. Đang chuẩn hóa các cột phân loại và văn bản...
2. Đang kiểm tra và lọc bỏ các bài review rác...
✔️ Đã loại bỏ 2124 dòng (do khách không chấm bất kỳ điểm dịch vụ nào).

📊 BÁO CÁO CHẤT LƯỢNG DỮ LIỆU AIRLINE REVIEWS
⚠️ Cảnh báo: Các cột sau vẫn còn ô NULL (CHỈ NÊN LÀ CỘT ĐIỂM SỐ RATING):
   - seat_comfort: trống 14101 dòng (9.1%)
   - cabin_staff_service: trống 14469 dòng (9.4%)
   - food_and_beverages: trống 42257 dòng (27.4%)
   - inflight_entertainment: trống 63280 dòng (41.0%)
   - ground_service: trống 41432 dòng (26.9%)
   - wifi_and_connectivity: trống 112999 dòng (73.3%)
   - value_for_money: trống 293 dòng (0.2%)
   - clean_keywords: trống 1 dòng (0.0%)
✔️ Kiểu dữ liệu ngày tháng: datetime64[ns]
✔️ Kiểu dữ liệu NPS (recommended): int64 (Các giá trị hiện có: [1 0])
-----------------------------------------------

In [15]:
import numpy as np
import pandas as pd

# Giả sử dataframe hiện tại của bạn là df (ví dụ: df = df_airline)

# 1. Chuẩn hóa cột date_submitted về định dạng Datetime
df["date_submitted"] = pd.to_datetime(df["date_submitted"], errors="coerce")

# 2. Xóa các review từ năm 2008 trở về trước (Chỉ giữ lại từ năm 2009 trở đi)
initial_rows = len(df)
df = df[df["date_submitted"].dt.year >= 2009].copy()
print(f"Đã xóa {initial_rows - len(df):,} dòng dữ liệu từ năm 2008 trở về trước.")
print(f"Số dòng còn lại: {len(df):,}")


# 3. Hàm phân loại 3 thời kỳ COVID
def categorize_covid_era(date_val):
    if pd.isna(date_val):
        return "Unknown"
    year = date_val.year
    if year <= 2019:
        return "Pre_covid"  # 2009 - 2019: Trước đại dịch
    elif year in [2020, 2021]:
        return "Covid"  # 2020 - 2021: Giai đoạn đóng cửa & khủng hoảng
    else:
        return "After_covid va phuc hoi"  # 2022 - Nay: Mở cửa & phục hồi


# Áp dụng tạo cột mới
df["data_era_covid"] = df["date_submitted"].apply(categorize_covid_era)

# Kiểm tra nhanh phân bổ số lượng review theo từng thời kỳ
print("\nPhân bổ dữ liệu theo thời kỳ:")
print(df["data_era_covid"].value_counts())

---

## Dataset Optimisation — Drop Pre-2009 & Text Columns

*Source notebook: `notebook_update_dataset.ipynb`*


In [17]:
#Note books dung de update dataset moi
## Xoa du lieu tu nam 2008 ve truoc
### Add them data era vu covid
### xoa di cot cleaned_keywords vi se dung ML tong hop keywords

In [18]:
import numpy as np
import pandas as pd

# Giả sử dataframe hiện tại của bạn là df (ví dụ: df = df_airline)

file_input = DATA_PROCESSED / "seat_reviews_PBI_Master_Optimized.csv"
file_output = "seat_reviews_PBI_Master_Optimized_V2.csv"

df = pd.read_csv(file_input)


# 1. Chuẩn hóa cột date_submitted về định dạng Datetime
df["date_submitted"] = pd.to_datetime(df["date_submitted"], errors="coerce")

# 2. Xóa các review từ năm 2008 trở về trước (Chỉ giữ lại từ năm 2009 trở đi)
initial_rows = len(df)
df = df[df["date_submitted"].dt.year >= 2009].copy()
print(f"Đã xóa {initial_rows - len(df):,} dòng dữ liệu từ năm 2008 trở về trước.")
print(f"Số dòng còn lại: {len(df):,}")


# Vector hóa bằng np.select (Nhanh hơn .apply gấp 10 lần)
years = df["date_submitted"].dt.year
conditions = [
    (years <= 2019),
    (years >= 2020) & (years <= 2021),
    (years >= 2022),
]
choices = ["Pre Covid", "Covid", "After Covid"]

df["data_era_covid"] = np.select(conditions, choices, default="Unknown")

# TỐI ƯU 2: Chuyển sang kiểu Category để giảm 80% dung lượng cột này khi nạp vào Power BI
df["data_era_covid"] = df["data_era_covid"].astype("category")

# Kiểm tra nhanh phân bổ số lượng review theo từng thời kỳ
print("\nPhân bổ dữ liệu theo thời kỳ:")
print(df["data_era_covid"].value_counts())



Đã xóa 92 dòng dữ liệu từ năm 2008 trở về trước.
Số dòng còn lại: 3,674

Phân bổ dữ liệu theo thời kỳ:
data_era_covid
Pre Covid      3154
After Covid     385
Covid           135
Name: count, dtype: int64


In [19]:
import gc
# 1. Danh sách các cột văn bản cần xóa (kiểm tra bao phủ cả 'cleaned_words', 'clean_keywords' và 'review' gốc nếu còn)
cols_to_drop = ["cleaned_words", "clean_keywords", "review"]

# Đo dung lượng bộ nhớ trước khi xóa
mem_before = df.memory_usage(deep=True).sum() / (1024**2)

# 2. Thực hiện xóa cột an toàn (chỉ xóa những cột thực sự tồn tại trong DataFrame)
existing_cols = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=existing_cols, errors="ignore")

# 3. Giải phóng bộ nhớ hệ thống ngay lập tức
gc.collect()

# Đo dung lượng bộ nhớ sau khi xóa
mem_after = df.memory_usage(deep=True).sum() / (1024**2)

print(f"✔️ Đã xóa các cột: {existing_cols}")
print(
    f"📐 Dung lượng dữ liệu giảm từ {mem_before:.2f} MB xuống {mem_after:.2f} MB (Tiết kiệm {(mem_before - mem_after):.2f} MB)"
)
print(f"📊 Kích thước bảng dữ liệu sẵn sàng cho Power BI: {df.shape}")

✔️ Đã xóa các cột: ['clean_keywords']
📐 Dung lượng dữ liệu giảm từ 3.06 MB xuống 2.01 MB (Tiết kiệm 1.05 MB)
📊 Kích thước bảng dữ liệu sẵn sàng cho Power BI: (3674, 26)


In [20]:
df.to_csv(file_output, index=False,encoding="utf-8-sig")